### Structured output
Models can be requested to provide their response in a format matching a given schema. This is useful for ensuring the output can be easily parsed and used in subsequent processing. LangChain supports multiple schema types and methods for enforcing structured output.

### Pydantic
Pydantic models provide the richest feature set with field validation, descriptions, and nested structures.

In [5]:
import os
from langchain.chat_models import init_chat_model

os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")

model=init_chat_model("groq:openai/gpt-oss-20b")

In [6]:
from pydantic import BaseModel,Field

class Movie(BaseModel):
    title:str=Field(description="The title of the movie")
    year:int=Field(description="This year the movie was released")
    director:str=Field(description="The director of the movie")
    rating:float=Field(description="The movies rating out of 10")

In [7]:
model_with_structure_output=model.with_structured_output(Movie)
model_with_structure_output

_ChatModelBinding(bound=ChatGroq(metadata={'lc_versions': {'langchain-core': '1.6.3', 'langchain': '1.4.0'}}, profile={'name': 'GPT OSS 20B', 'release_date': '2025-08-05', 'last_updated': '2026-05-27', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x0000022EB70C5FD0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x0000022EB70D8DA0>, model_name='openai/gpt-oss-20b', model_kwargs={}, groq_api_key=SecretStr('**********')), kwargs={'tools': [{'type': 'function', 'function': {'name': 'Movie', 'description': '', 'parameters': {'properties': {'title': {'description': 'The titl

In [8]:
model.invoke("Provide details about the moview Inception")



AIMessage(content='**Inception**  \n*Released: July\u202f16,\u202f2010 (United States)*  \n*Director: Christopher\u202fNolan*  \n*Screenplay: Christopher\u202fNolan*  \n*Producer: Emma\u202fThomas, Christopher\u202fNolan, Jonathan\u202fC.\u202fKahn*  \n*Genre: Science‑fiction, Action, Thriller*  \n*Runtime: 2\u202fh\u202f28\u202fmin*  \n*Language: English*  \n*Budget: ~\u202f$160\u202fmillion*  \n*Worldwide gross: ~$830\u202fmillion*  \n\n---\n\n## 1. Premise / Plot\n\n*Inception* follows Dom\u202fCobb (Leonardo DiCaprio), a “extractor” who enters people’s dreams to steal secrets. He is offered a chance to erase his criminal record: perform an *inception*—plant an idea in a target’s mind. Cobb assembles a team:\n\n| Member | Role | Key skills |\n|--------|------|------------|\n| Dom\u202fCobb | Lead extractor | Master of dream‑navigation |\n| Arthur | Planner, second‑in‑command | Meticulous, time‑keeping |\n| Ariadne | Architect | Builds dream environments |\n| Eames | Forger | Shape‑s

In [10]:
response=model_with_structure_output.invoke("Provide details about the moview Inception")
response

Movie(title='Inception', year=2010, director='Christopher Nolan', rating=8.8)


# MEssage output alongside parsed structure

In [12]:
from pydantic import BaseModel, Field

class Movie(BaseModel):
    """A movie with details."""
    title: str = Field(..., description="The title of the movie")
    year: int = Field(..., description="The year the movie was released")
    director: str = Field(..., description="The director of the movie")
    rating: float = Field(..., description="The movie's rating out of 10")

model_with_structure = model.with_structured_output(Movie, include_raw=True)  

response = model_with_structure.invoke("Provide details about the movie Inception")
response

{'raw': AIMessage(content='', additional_kwargs={'reasoning_content': 'The user wants details about the movie Inception. We need to use the function "Movie" to get details. The function signature expects director, rating, title, year. We should call the function with those arguments. The function likely returns details about the movie. We need to call it.\n\nWe will call functions.Movie with director="Christopher Nolan", rating=8.8 (approx), title="Inception", year=2010.', 'tool_calls': [{'id': 'fc_06da72ed-ffc4-4969-87c6-226eeb7625b8', 'function': {'arguments': '{"director":"Christopher Nolan","rating":8.8,"title":"Inception","year":2010}', 'name': 'Movie'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 128, 'prompt_tokens': 164, 'total_tokens': 292, 'completion_time': 0.191436517, 'completion_tokens_details': {'reasoning_tokens': 89}, 'prompt_time': 0.009995948, 'prompt_tokens_details': None, 'queue_time': 0.341063658, 'total_time': 0.201432465}, 'mod

# Nested Structure

In [ ]:
from pydantic import BaseModel,Field

class Actor(BaseModel):
    name:str
    role:str


class Movie_details(BaseModel):
    title: str
    year: int
    cast:list[Actor]
    genres:list[str]
    budget:float | None = Field(None,description="budget in million USD")

model_with_structure=model.with_structured_output(Movie_details)
response = model_with_structure.invoke("Provide details about the movie Inception")
response

Movie_details(title='Inception', year=2010, cast=[Actor(name='Leonardo DiCaprio', role='Dom Cobb'), Actor(name='Joseph Gordon-Levitt', role='Arthur'), Actor(name='Ellen Page', role='Ariadne'), Actor(name='Tom Hardy', role='Eames'), Actor(name='Ken Watanabe', role='Saito'), Actor(name='Cillian Murphy', role='Robert Fischer')], genres=['Action', 'Adventure', 'Sci-Fi', 'Thriller'], budget=160000000.0)

# TypedDict
TypedDict provides a simpler alternative using Python’s built-in typing, ideal when you don’t need runtime validation.

In [18]:
from typing_extensions import TypedDict,Annotated

class MovieDict(TypedDict):
    """A movie with details."""
    title: Annotated[str, ..., "The title of the movie"]
    year: Annotated[int, ..., "The year the movie was released"]
    director: Annotated[str, ..., "The director of the movie"]
    rating: Annotated[float, ..., "The movie's rating out of 10"]


model_withtypedict=model.with_structured_output(MovieDict)
response=model_withtypedict.invoke("Please provide the details of the movie avengers")
response

{'director': 'Anthony Russo & Joe Russo',
 'rating': 8.4,
 'title': 'Avengers: Endgame',
 'year': 2019}

In [20]:

class Actor(TypedDict):
    name:str
    role:str


class Movie_details(TypedDict):
    title: str
    year: int
    cast:list[Actor]
    genres:list[str]
    budget:float | None = Field(None,description="budget in million USD")

model_with_structure=model.with_structured_output(Movie_details)
response = model_with_structure.invoke("Provide details about the movie Inception")
response

{'budget': 160000000,
 'cast': [{'name': 'Leonardo DiCaprio', 'role': 'Dom Cobb'},
  {'name': 'Joseph Gordon-Levitt', 'role': 'Arthur'},
  {'name': 'Elliot Page', 'role': 'Ariadne'},
  {'name': 'Tom Hardy', 'role': 'Eames'},
  {'name': 'Ken Watanabe', 'role': 'Saito'},
  {'name': 'Cillian Murphy', 'role': 'Robert Fischer'},
  {'name': 'Michael Caine', 'role': 'Professor Stephen Miles'},
  {'name': 'Marion Cotillard', 'role': 'Mal'}],
 'genres': ['Action', 'Science Fiction', 'Thriller'],
 'title': 'Inception',
 'year': 2010}

In [21]:
model.profile

{'name': 'GPT OSS 20B',
 'release_date': '2025-08-05',
 'last_updated': '2026-05-27',
 'open_weights': True,
 'max_input_tokens': 131072,
 'max_output_tokens': 65536,
 'text_inputs': True,
 'image_inputs': False,
 'audio_inputs': False,
 'video_inputs': False,
 'text_outputs': True,
 'image_outputs': False,
 'audio_outputs': False,
 'video_outputs': False,
 'reasoning_output': True,
 'tool_calling': True,
 'structured_output': True,
 'attachment': False,
 'temperature': True}

# DataClasses
A data class is a class typically containing mainly data, although there aren’t really any restrictions. You create it using the @dataclass decorator

In [23]:
from pydantic import BaseModel, Field
from langchain.agents import create_agent


class ContactInfo(BaseModel):
    """Contact information for a person."""
    name: str = Field(description="The name of the person")
    email: str = Field(description="The email address of the person")
    phone: str = Field(description="The phone number of the person")

agent = create_agent(
    model=model,
    response_format=ContactInfo  # Auto-selects ProviderStrategy
)

result = agent.invoke({
    "messages": [{"role": "user", "content": "Extract contact info from: John Doe, john@example.com, (555) 123-4567"}]
})

result

{'messages': [HumanMessage(content='Extract contact info from: John Doe, john@example.com, (555) 123-4567', additional_kwargs={}, response_metadata={}, id='2b5ebaab-74b7-4ffe-832d-511985593d2f'),
  AIMessage(content='{"name":"John Doe","email":"john@example.com","phone":"(555) 123-4567"}', additional_kwargs={'reasoning_content': 'We need to output a JSON object according to the ContactInfo schema. Fields: name, email, phone. Must include all. Provide compact JSON. Ensure valid JSON.\n\nThe contact info: name: "John Doe", email: "john@example.com", phone: "(555) 123-4567"\n\nReturn JSON: {"name":"John Doe","email":"john@example.com","phone":"(555) 123-4567"}'}, response_metadata={'token_usage': {'completion_tokens': 125, 'prompt_tokens': 238, 'total_tokens': 363, 'completion_time': 0.157686906, 'completion_tokens_details': {'reasoning_tokens': 89}, 'prompt_time': 0.012298169, 'prompt_tokens_details': None, 'queue_time': 0.248340364, 'total_time': 0.169985075}, 'model_name': 'openai/gpt-

In [ ]:
result["structured_response"] 

ContactInfo(name='John Doe', email='john@example.com', phone='(555) 123-4567')

In [25]:
## Dataclass

from dataclasses import dataclass
from langchain.agents import create_agent

@dataclass
class ContactInfo:
    """Contact information for a person."""
    name: str # The name of the person
    email: str # The email address of the person
    phone: str # The phone number of the person


agent = create_agent(
    model=model,
    response_format=ContactInfo  # Auto-selects ProviderStrategy
)

result = agent.invoke({
    "messages": [{"role": "user", "content": "Extract contact info from: John Doe, john@example.com, (555) 123-4567"}]
})

result["structured_response"]

ContactInfo(name='John Doe', email='john@example.com', phone='(555) 123-4567')